# 📊 Week 1 — Exploratory Data Analysis (EDA)
**Smart E-Commerce Analytics Platform**

**Dataset:** Olist Brazilian E-Commerce Public Dataset (Kaggle)
- 9 relational CSV files
- ~100k orders from 2016–2018
- Real Brazilian e-commerce data

**This notebook covers:**
1. Load all 9 raw Olist CSV files
2. Inspect shapes, dtypes, missing values
3. Data cleaning & merging
4. Univariate & bivariate analysis
5. Sales trends, top categories, geographic analysis
6. Payment & review analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

OLIST = '../data/olist/'
print('Libraries loaded ✅')

## 1. Load Raw Olist Datasets

In [ ]:
orders    = pd.read_csv(OLIST + 'olist_orders_dataset.csv',
                        parse_dates=['order_purchase_timestamp',
                                     'order_delivered_customer_date',
                                     'order_estimated_delivery_date'])
items     = pd.read_csv(OLIST + 'olist_order_items_dataset.csv')
payments  = pd.read_csv(OLIST + 'olist_order_payments_dataset.csv')
customers = pd.read_csv(OLIST + 'olist_customers_dataset.csv')
products  = pd.read_csv(OLIST + 'olist_products_dataset.csv')
sellers   = pd.read_csv(OLIST + 'olist_sellers_dataset.csv')
reviews   = pd.read_csv(OLIST + 'olist_order_reviews_dataset.csv')
geo       = pd.read_csv(OLIST + 'olist_geolocation_dataset.csv')
trans     = pd.read_csv(OLIST + 'product_category_name_translation.csv')

print('Dataset shapes:')
for name, df in [('orders',orders),('items',items),('payments',payments),
                  ('customers',customers),('products',products),
                  ('sellers',sellers),('reviews',reviews),('geo',geo),('trans',trans)]:
    print(f'  {name:12s}: {df.shape}')

## 2. Data Inspection — Missing Values & Dtypes

In [ ]:
print('=== ORDERS ===')
display(orders.head(3))
print('\nDtypes:')
print(orders.dtypes)
print('\nMissing values:')
miss = orders.isnull().sum()
print(miss[miss > 0])

In [ ]:
print('=== ORDER STATUS DISTRIBUTION ===')
print(orders['order_status'].value_counts())

print('\n=== PAYMENT TYPES ===')
print(payments['payment_type'].value_counts())

print('\n=== REVIEW SCORES ===')
print(reviews['review_score'].value_counts().sort_index())

In [ ]:
print('=== PRODUCTS — Missing Values ===')
miss_prod = products.isnull().sum()
print(miss_prod[miss_prod > 0])

print('\n=== CUSTOMERS — Top States ===')
print(customers['customer_state'].value_counts().head(10))

## 3. Data Cleaning & Merging

In [ ]:
# Step 1: Translate product categories (Portuguese → English)
products = products.merge(trans, on='product_category_name', how='left')
products['category'] = products['product_category_name_english'].fillna(
    products['product_category_name'].fillna('unknown'))
products['category'] = products['category'].str.replace('_', ' ').str.title()
print(f'Products with English category: {products["category"].notna().sum()}')

# Step 2: Aggregate payment totals per order
pay_total = payments.groupby('order_id')['payment_value'].sum().reset_index()
pay_total.columns = ['order_id', 'total_amount']
print(f'Payment aggregation: {len(pay_total)} orders')

# Step 3: Aggregate items per order
items_agg = items.groupby('order_id').agg(
    quantity   = ('order_item_id', 'count'),
    product_id = ('product_id',    'first')
).reset_index()
print(f'Items aggregation: {len(items_agg)} orders')

# Step 4: Average review score per order
rev_score = reviews.groupby('order_id')['review_score'].mean().reset_index()
print(f'Review aggregation: {len(rev_score)} orders')

In [ ]:
# Step 5: Merge all tables
df = (orders
      .merge(customers[['customer_id','customer_unique_id','customer_city','customer_state']],
             on='customer_id', how='left')
      .merge(pay_total,  on='order_id', how='left')
      .merge(items_agg,  on='order_id', how='left')
      .merge(products[['product_id','category']], on='product_id', how='left')
      .merge(rev_score,  on='order_id', how='left'))

print(f'Merged shape: {df.shape}')
print(f'\nOrder status counts:')
print(df['order_status'].value_counts())

In [ ]:
# Step 6: Filter to delivered orders only
before = len(df)
df = df[df['order_status'] == 'delivered'].copy()
print(f'After filtering to delivered: {len(df):,} / {before:,} ({len(df)/before*100:.1f}%)')

# Step 7: Drop rows with missing total_amount or timestamp
df = df.dropna(subset=['total_amount', 'order_purchase_timestamp'])
print(f'After dropping nulls: {len(df):,}')

# Step 8: Fill remaining nulls
df['category']     = df['category'].fillna('Unknown')
df['review_score'] = df['review_score'].fillna(3.0)
df['quantity']     = df['quantity'].fillna(1).astype(int)

# Step 9: Rename columns
df = df.rename(columns={
    'customer_unique_id':       'customer_id',
    'order_purchase_timestamp': 'transaction_date',
    'customer_city':            'city',
    'customer_state':           'state',
})

# Step 10: Add time features
df['order_year']   = df['transaction_date'].dt.year
df['order_month']  = df['transaction_date'].dt.month
df['day_of_week']  = df['transaction_date'].dt.day_name()
df['hour']         = df['transaction_date'].dt.hour

print(f'\nFinal cleaned shape: {df.shape}')
print(f'Missing values remaining:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Summary statistics
print('=== CLEANED DATA SUMMARY ===')
display(df[['total_amount','quantity','review_score']].describe().round(2))
display(df.head(3))

## 4. Sales Trend Analysis

In [ ]:
monthly = df.groupby(df['transaction_date'].dt.to_period('M'))['total_amount'].sum().reset_index()
monthly['transaction_date'] = monthly['transaction_date'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly['transaction_date'], monthly['total_amount'], alpha=0.2, color='steelblue')
ax.plot(monthly['transaction_date'], monthly['total_amount'],
        marker='o', linewidth=2, color='steelblue', markersize=5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
plt.xticks(rotation=45)
ax.set_title('Monthly Revenue Trend — Olist 2016–2018')
ax.set_ylabel('Revenue (R$)')
plt.tight_layout()
plt.savefig('../report/monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak month: {monthly.loc[monthly["total_amount"].idxmax(), "transaction_date"].strftime("%b %Y")}')
print(f'Total revenue: R${df["total_amount"].sum():,.0f}')

## 5. Revenue by Category

In [ ]:
cat_rev = df.groupby('category')['total_amount'].sum().sort_values(ascending=False)
top10_cat = cat_rev.head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top10_cat.plot(kind='barh', ax=axes[0], color=sns.color_palette('Blues_r', 10))
axes[0].set_title('Top 10 Categories by Revenue')
axes[0].set_xlabel('Revenue (R$)')
axes[0].invert_yaxis()

axes[1].pie(top10_cat.values, labels=[c[:20] for c in top10_cat.index],
            autopct='%1.1f%%', colors=sns.color_palette('muted', 10), startangle=140)
axes[1].set_title('Revenue Share — Top 10 Categories')

plt.tight_layout()
plt.savefig('../report/category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Number of unique categories: {df["category"].nunique()}')

## 6. Geographic Analysis — Top States

In [ ]:
state_orders = df['state'].value_counts().head(10)
state_rev    = df.groupby('state')['total_amount'].sum().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

state_orders.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', 10), edgecolor='white')
axes[0].set_title('Top 10 States by Order Count')
axes[0].set_ylabel('Orders')
axes[0].tick_params(axis='x', rotation=30)

state_rev.plot(kind='bar', ax=axes[1], color=sns.color_palette('Blues_r', 10), edgecolor='white')
axes[1].set_title('Top 10 States by Revenue')
axes[1].set_ylabel('Revenue (R$)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../report/state_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Order Value Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(df['total_amount'], bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Order Value Distribution (Full)')
axes[0].set_xlabel('Order Value (R$)')
axes[0].set_ylabel('Count')

# Capped at 99th percentile
cap = df['total_amount'].quantile(0.99)
axes[1].hist(df['total_amount'].clip(upper=cap), bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title(f'Order Value Distribution (capped at R${cap:.0f})')
axes[1].set_xlabel('Order Value (R$)')

plt.tight_layout()
plt.savefig('../report/order_value_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean order value:   R${df["total_amount"].mean():.2f}')
print(f'Median order value: R${df["total_amount"].median():.2f}')
print(f'Max order value:    R${df["total_amount"].max():.2f}')

## 8. Payment & Review Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Payment types
pay_cnt = payments['payment_type'].value_counts()
pay_cnt.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', len(pay_cnt)), edgecolor='white')
axes[0].set_title('Payment Types (All Orders)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Review scores
rev_cnt = reviews['review_score'].value_counts().sort_index()
colors  = ['#d73027','#f46d43','#fdae61','#a6d96a','#1a9850']
rev_cnt.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Review Score Distribution')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../report/payment_reviews.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Average review score: {reviews["review_score"].mean():.2f}')
print(f'5-star reviews: {(reviews["review_score"]==5).sum():,} ({(reviews["review_score"]==5).mean()*100:.1f}%)')

## 9. Order Timing Analysis

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_cnt   = df['day_of_week'].value_counts().reindex(dow_order)
hour_cnt  = df.groupby('hour')['order_id'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dow_cnt.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Orders by Day of Week')
axes[0].set_ylabel('Orders')
axes[0].tick_params(axis='x', rotation=30)

axes[1].plot(hour_cnt.index, hour_cnt.values, color='steelblue', linewidth=2, marker='o', markersize=5)
axes[1].fill_between(hour_cnt.index, hour_cnt.values, alpha=0.2, color='steelblue')
axes[1].set_title('Orders by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Orders')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('../report/order_timing.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Correlation Heatmap

In [ ]:
num_cols = ['total_amount', 'quantity', 'review_score']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, square=True)
ax.set_title('Correlation Heatmap — Order Features')
plt.tight_layout()
plt.savefig('../report/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ EDA Complete!')
print(f'Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df["transaction_date"].min().date()} → {df["transaction_date"].max().date()}')
print(f'Unique customers: {df["customer_id"].nunique():,}')
print(f'Total revenue: R${df["total_amount"].sum():,.0f}')